In [ ]:
import numpy as np
import pandas as pd 
import tensorflow as tf
import re
import html
import ftfy
import emoji
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Bidirectional, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
!pip install -q evaluate


In [ ]:
np.random.seed(42) 
tf.random.set_seed(42)
MAX_VOCAB=40000 
MAX_LEN=200
BATCH_SIZE=128
EPOCHS_RNN=4 
EPOCHS_LSTM=8
EMBED_DIM=128
RNN_UNITS=64

In [ ]:
langs=['en','ru','uk','de','es','am','zh','ar','hi','it','fr','he','hin','tt','ja']
dfs=[]
for lg in langs:
    df=load_dataset("textdetox/multilingual_toxicity_dataset",split=lg).to_pandas()
    df["lang"]=lg; dfs.append(df)
df_full=pd.concat(dfs,ignore_index=True)


In [ ]:
possible_cols=["toxicity","label","toxic","target","binary"]
for c in possible_cols:
    if c in df_full.columns:
        tox_col=c; break
df_full["label"]=(df_full[tox_col]>=0.5).astype(int) if df_full[tox_col].dtype==float else df_full[tox_col].astype(int)
u=re.compile(r"http\S+|www\.\S+"); h=re.compile(r"<.*?>")
def clean(t):
    if not isinstance(t,str): return ""
    t=ftfy.fix_text(t); t=html.unescape(t); t=u.sub(" URL ",t); t=h.sub(" ",t); t=emoji.replace_emoji(t,replace=" EMOJI "); return t.strip()
text_col="text" if "text" in df_full.columns else df_full.columns[0]
df_full["clean"]=df_full[text_col].map(clean)
df_val=df_full.sample(frac=0.1,random_state=42)
df_train=df_full.drop(df_val.index)


In [ ]:
# dataset visualisation: label balance
lab=df_full["label"].value_counts().sort_index()
plt.bar(["non-toxic","toxic"],lab.values,color=["skyblue","salmon"])
plt.yscale("log"); plt.title("label distribution (all languages)")
plt.ylabel("count (log)")
for i,v in enumerate(lab.values): plt.text(i,v*1.1,str(v),ha="center")
plt.show()
print("toxic % =",100*lab.get(1,0)/lab.sum())


In [ ]:
# dataset visualisation: examples per language
lc=df_full["lang"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(10,4))
plt.bar(lc.index,lc.values,color="mediumpurple")
plt.yscale("log"); plt.xticks(rotation=45,ha="right")
plt.title("examples per language"); plt.ylabel("count (log)")
plt.show()


In [ ]:
tok=Tokenizer(num_words=MAX_VOCAB,oov_token="<OOV>")
tok.fit_on_texts(df_train["clean"])
X_train=pad_sequences(tok.texts_to_sequences(df_train["clean"]),maxlen=MAX_LEN,padding="post",truncating="post")
y_train=df_train["label"].values.astype(np.float32)
X_val=pad_sequences(tok.texts_to_sequences(df_val["clean"]),maxlen=MAX_LEN,padding="post",truncating="post")
y_val=df_val["label"].values.astype(np.float32); val_lang=df_val["lang"].values
cw_vals=compute_class_weight(class_weight="balanced",classes=np.array([0,1]),y=y_train)
class_w={0:float(cw_vals[0]),1:float(cw_vals[1])}


In [ ]:
# dataset visualisation: comment length (words)
lens=df_full["clean"].astype(str).str.split().map(len)
plt.hist(lens,bins=60,color="goldenrod",log=True)
plt.xlim(0,300); plt.title("comment length distribution")
plt.xlabel("words"); plt.ylabel("count (log)")
plt.show()
print("mean",lens.mean(),"std",lens.std())


In [ ]:
rnn=Sequential([Embedding(MAX_VOCAB,EMBED_DIM,input_length=MAX_LEN),
                SimpleRNN(RNN_UNITS,dropout=0.2),
                Dense(1,activation="sigmoid")])
rnn.compile(loss="binary_crossentropy",optimizer=tf.keras.optimizers.Adam(1e-3),metrics=["accuracy"])
rnn.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=EPOCHS_RNN,
        batch_size=BATCH_SIZE,class_weight=class_w,
        callbacks=[EarlyStopping(monitor="val_loss",patience=2,restore_best_weights=True),
                   ModelCheckpoint("best_rnn.weights.h5",monitor="val_loss",
                                   save_best_only=True,save_weights_only=True)],
        verbose=2)


In [ ]:
rnn.load_weights("best_rnn.weights.h5")
test_lines=["Das ist absoluter Müll.","Vielen Dank für den schnellen Versand.","Fuck off"]
seqs=pad_sequences(tok.texts_to_sequences([clean(t) for t in test_lines]),
                   maxlen=MAX_LEN,padding="post",truncating="post")
probs=rnn.predict(seqs).flatten()
for t,p in zip(test_lines,probs): print(f"{p:5.3f}",("toxic" if p>=0.5 else "non-toxic"),"|",t)


In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau

lstm = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    Dropout(0.2),                                  # embedding dropout
    Bidirectional(LSTM(RNN_UNITS,
                       dropout=0.3,
                       recurrent_dropout=0.3)),
    Dense(1, activation="sigmoid")
])

lstm.compile(loss="binary_crossentropy",
             optimizer=tf.keras.optimizers.Adam(1e-3),
             metrics=["accuracy",
                      tf.keras.metrics.AUC(name="auc")])

callbacks = [
    EarlyStopping(monitor="val_auc", mode="max",
                  patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_auc", mode="max",
                      factor=0.5, patience=1, min_lr=1e-5),
    ModelCheckpoint("best_lstm.weights.h5",
                    monitor="val_auc", mode="max",
                    save_best_only=True, save_weights_only=True)
]

lstm.fit(X_train, y_train,
         validation_data=(X_val, y_val),
         epochs=15,
         batch_size=BATCH_SIZE,
         class_weight={0: cw[0], 1: cw[1]*1.5},
         callbacks=callbacks,
         verbose=2)


In [ ]:
lstm.load_weights("best_lstm.weights.h5")
lstm.save("NoisyBazaar.keras")


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

model = tf.keras.models.load_model("NoisyBazaar.keras") 

def predict_toxic(sentences, tok, clean_fn, max_len=MAX_LEN):
    seq = tok.texts_to_sequences([clean_fn(s) for s in sentences])
    X   = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    probs = model.predict(X).flatten()
    return probs, (probs >= 0.55).astype(int)

test_lines = [
     # English toxic
    "You’re a total waste of oxygen—don’t ever message me again.",
    "I hope your scammy little shop burns to the ground.",
    "Get lost, moron; nobody cares about your trash product.",
    "Stop spamming me, idiot—learn to read the listing.",
    "Only a fool like you would think that price is fair.",
    # English non-toxic
    "Thanks for answering my questions so quickly!",
    "Could we schedule pickup on Saturday morning?",
    "Is the warranty still valid for another year?",
    "Appreciate the detailed photos—looks great.",
    "Sounds good; I’ll send payment as soon as I get home.",
    # German toxic
    "Du bist echt das Letzte, niemand braucht deinen Schrott.",
    "Hau ab, Spinner—verschwendest nur meine Zeit.",
    "Nur ein Vollidiot wie du verlangt so einen Preis.",
    "Ich hoffe, dein Laden geht bald pleite.",
    "Dein Angebot ist der reinste Müll, behalt’s für dich.",
    # German non-toxic
    "Vielen Dank für die schnelle Rückmeldung!",
    "Kannst du den Artikel bis morgen reservieren?",
    "Gibt es noch Garantie auf das Gerät?",
    "Der Preis klingt fair; lass uns am Freitag treffen.",
    "Nochmals danke für deine Hilfe beim Einrichten."
]

probs, labels = predict_toxic(test_lines, tok, clean)
for t,p,l in zip(test_lines, probs, labels):
    print(f"{p:5.3f} → {'toxic' if l else 'non-toxic'} | {t}")


In [ ]:

lstm = tf.keras.models.load_model("NoisyBazaar.keras", compile=False)
lstm.save("NoisyBazaarUP.h5", include_optimizer=False)

In [ ]:
from cryptography.fernet import Fernet, os 
print(Fernet.generate_key().decode())